# 01 — Data Collection
**BUSI 1783 Individual Project — Jay Panchal (001495232)**  
**Supervisor:** Dr. Martina Testori

This notebook collects the two raw data sources for the study: daily Bitcoin price data and the archive of r/Bitcoin posts. The study window is 1 January 2022 to 31 December 2025, split at the first US spot Bitcoin ETF trading day (10 January 2024) into a pre-ETF and post-ETF regime.

**Note on data sources.** The proposal specified CoinGecko (prices) and PRAW (Reddit). Both were changed during implementation for documented, practical reasons: CoinGecko's free tier only serves the past 365 days, and PRAW cannot retrieve historical posts at scale. Prices were therefore obtained from Yahoo Finance via `yfinance`, and historical Reddit posts from the Arctic Shift archive (the community-maintained successor to Pushshift). The research window, variables and analysis are unchanged.

## 1.1 Environment check
Confirm the required libraries are installed and importable.

In [1]:
# Test all libraries are installed
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import praw
from pycoingecko import CoinGeckoAPI
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("pandas:", pd.__version__)
print("statsmodels:", sm.__version__)
print("All libraries loaded successfully ✅")

pandas: 3.0.0
statsmodels: 0.14.6
All libraries loaded successfully ✅


## 1.2 Bitcoin price data (Yahoo Finance via yfinance)
Daily Bitcoin closing prices for the study window are downloaded from Yahoo Finance. Only the closing price is retained.

In [2]:
import yfinance as yf
import pandas as pd

# Download Bitcoin prices from Yahoo Finance
print("Fetching Bitcoin price data... please wait")

btc = yf.download('BTC-USD', start='2022-01-01', end='2025-12-31')

# Keep only the closing price
prices = btc[['Close']].reset_index()
prices.columns = ['date', 'price']
prices['date'] = pd.to_datetime(prices['date'])

print(f"✅ Done! Downloaded {len(prices)} days of Bitcoin price data")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
print(prices.head(10))

Fetching Bitcoin price data... please wait


C:\Users\asus\AppData\Local\Temp\ipykernel_17200\3144416791.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc = yf.download('BTC-USD', start='2022-01-01', end='2025-12-31')
C:\Users\asus\AppData\Roaming\Python\Python313\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  1 of 1 completed

✅ Done! Downloaded 1460 days of Bitcoin price data
Date range: 2022-01-01 00:00:00 to 2025-12-30 00:00:00
        date         price
0 2022-01-01  47686.812500
1 2022-01-02  47345.218750
2 2022-01-03  46458.117188
3 2022-01-04  45897.574219
4 2022-01-05  43569.003906
5 2022-01-06  43160.929688
6 2022-01-07  41557.902344
7 2022-01-08  41733.941406
8 2022-01-09  41911.601562
9 2022-01-10  41821.261719


Save the raw price series to the project data folder.

In [3]:
# Save raw price data to your data folder
prices.to_csv(r'C:\Users\asus\BUSI1783-Project\data\raw\bitcoin_prices_raw.csv', 
              index=False)

print("✅ Bitcoin price data saved!")
print(f"Total rows: {len(prices)}")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")

✅ Bitcoin price data saved!
Total rows: 1460
Date range: 2022-01-01 to 2025-12-30


## 1.3 Reddit posts (Arctic Shift archive)
Historical r/Bitcoin posts were downloaded from the Arctic Shift archive as a compressed JSON-lines file. The cell below inspects the structure of a single record to confirm the fields used later (title, body, timestamp, score).

In [4]:
import json

filepath = r'C:\Users\asus\BUSI1783-Project\data\raw\r_Bitcoin_posts.jsonl'

# Read just the first post to see its structure
with open(filepath, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    first_post = json.loads(first_line)

# Show all available fields
print("Available fields in each post:")
for key in first_post.keys():
    print(f"  - {key}")

print("\n--- Sample values ---")
print("Title:", first_post.get('title', 'N/A'))
print("Created (timestamp):", first_post.get('created_utc', 'N/A'))
print("Selftext (body):", str(first_post.get('selftext', 'N/A'))[:100])
print("Score:", first_post.get('score', 'N/A'))

Available fields in each post:
  - all_awardings
  - allow_live_comments
  - archived
  - author
  - author_created_utc
  - author_flair_background_color
  - author_flair_css_class
  - author_flair_template_id
  - author_flair_text
  - author_flair_text_color
  - awarders
  - banned_by
  - can_gild
  - can_mod_post
  - category
  - content_categories
  - contest_mode
  - created_utc
  - discussion_type
  - distinguished
  - domain
  - edited
  - gilded
  - gildings
  - hidden
  - hide_score
  - id
  - is_created_from_ads_ui
  - is_crosspostable
  - is_meta
  - is_original_content
  - is_reddit_media_domain
  - is_robot_indexable
  - is_self
  - is_video
  - link_flair_background_color
  - link_flair_css_class
  - link_flair_richtext
  - link_flair_template_id
  - link_flair_text
  - link_flair_text_color
  - link_flair_type
  - locked
  - media
  - media_embed
  - media_only
  - name
  - no_follow
  - num_comments
  - num_crossposts
  - over_18
  - parent_whitelist_status
  - permalink

---
*End of data collection. Raw prices are saved to `data/raw/bitcoin_prices_raw.csv`; the raw Reddit dump (`data/raw/r_Bitcoin_posts.jsonl`) is retained locally and is too large for the repository — it is available on request.*